# Step 0 — Detector calibration from a CeO2 pattern, from scratch

This is the calibration step every other notebook in this directory depends on: the
sample-to-detector distance, beam center, and detector tilts that
`05_raster_lattice_single_point.ipynb` and `06_raster_lattice_batch.ipynb` take as a fixed
`Geometry`. Get this wrong and you don't get "no answer" -- you get a *confident, wrong* one,
because a small geometry error can look exactly like a real lattice distortion.

**The hard rule: calibrate from scratch, every time -- no seed of any kind that isn't computed
by the code from the image itself.** Never seed from a previously-converged geometry, "the
value we used last time," or a copied `.poni` file. This notebook calls
`midas_calibrate_v2.pipelines.auto.calibrate()` with **no seed arguments at all**: with none
supplied, `calibrate()` seeds itself internally via its own validated automated seeder,
`midas_calibrate_v2.seed.auto_seed.make_seed`, and records that choice in the result
(`seed_method`) -- verified below, not assumed.

**No material or path default anywhere** -- same discipline as `05`/`06`'s intro. Set `IMG_PATH`
to your own CeO2 (or other known-cubic-calibrant) exposure below.

## Step 0 — CONFIG

Set your calibrant image, material, wavelength, and detector spec -- no defaults are silently
assumed. The gates in the next cell are what actually decide whether to trust the result; read
them every time, not just the converged numbers.

In [ ]:
import os, time
from pathlib import Path
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import numpy as np
import tifffile

# ---- the only lines you change ------------------------------------------------------
IMG_PATH = None                 # YOUR CeO2 (or other cubic calibrant) exposure, a .tif
CALIBRANT_A = 5.4116            # YOUR calibrant's cubic cell edge, Angstrom (CeO2: 5.4116)
CALIBRANT_SPACE_GROUP = 225     # YOUR calibrant's space group (CeO2, Fm-3m: 225)
WAVELENGTH_A = 0.42459          # YOUR beamline wavelength, Angstrom
PIXEL_SIZE_UM = 172.0           # YOUR detector pixel pitch, micron (assumed isotropic)
# No distance/beam-center guess of any kind belongs here -- not even a rough nominal setup
# distance. That is exactly the prior information this test exists to NOT lean on.
# ---------------------------------------------------------------------------------------

if IMG_PATH is None:
    raise FileNotFoundError(
        "Set IMG_PATH to your own CeO2 (or other known-cubic-calibrant) exposure -- "
        "there is no material default and no fallback path in this notebook."
    )
IMG_PATH = Path(IMG_PATH)

raw = tifffile.imread(str(IMG_PATH)).astype(np.float64)
print("image shape:", raw.shape, "dtype:", raw.dtype, "min/max:", raw.min(), raw.max())
mask = (raw < 0)   # Pilatus convention: pixel < 0 marks a gap/dead pixel -- confirm this
                    # matches YOUR detector before trusting it on a different one.
print(f"masked pixels: {mask.sum()} ({100*mask.mean():.2f}%)")
img = np.where(mask, 0, raw)


## Step 1 — Calibrate from scratch

`calibrate()`, given no seed arguments, calls its own validated automated seeder
(`make_seed`, Hough + arc/chord fitting on the ring pattern) internally, then refines
distance, beam-center, tilts, and the distortion harmonics, checking itself against
reliability gates throughout.

**A different entry point in this same package,
`midas_calibrate_v2.pipelines.first_time.first_time_calibrate`, was tried on a real,
heavily-masked detector image for this project and was found to walk away from a good,
identical `make_seed` seed into a much worse basin** (confirmed by a ring overlay drifting
visibly off the real rings, Step 3 below) -- while `calibrate()`, from the same starting
point, stayed close to it and converged well. That is a real difference between two pipelines
in this package, not between two seeds; if your own calibration doesn't converge with
`calibrate()`, that is the more informative thing to report than switching seeders.

In [ ]:
from midas_calibrate_v2 import calibrate

t0 = time.time()
res = calibrate(
    img,
    wavelength=WAVELENGTH_A,
    pxY=PIXEL_SIZE_UM,
    mask=mask,
    calibrant={"a": CALIBRANT_A, "sg": CALIBRANT_SPACE_GROUP},
    # No BC_guess, no initial_Lsd, no initial_BC_y/z -- calibrate() seeds itself.
    verbose=True,
)
print(f"\nelapsed: {time.time()-t0:.1f}s")
print(f"\nseed_method = {res.seed_method!r}   ({res.seed_note})")
if res.seed_method != "make_seed":
    print(f"WARNING: seed_method is {res.seed_method!r}, not the validated 'make_seed' -- "
         "a 'fallback' geometry is NOT of the same quality and should not be trusted the same way.")


## Step 2 — Read the result: the gates, not just the numbers

`AutoCalibrationResult` carries the converged geometry AND the reliability-gate report. Print
every gate's verdict, not just the convenient ones -- but also don't over-read a small overage
on `strain_cap` as a failed calibration; look at the ring overlay in Step 3 to judge that, not
the gate label alone:

- **`seed_provenance`** -- auto-derived from the image, or supplied by a person? A person-supplied
  seed forfeits the "from scratch" guarantee.
- **`strain_cap`** -- residual pseudo-strain (`|1 - R_obs/R_pred|`, in microstrain) vs. a
  material-appropriate cap (100 microstrain default, a conservative generic setting). This is a
  FRACTIONAL quantity: for a fixed absolute centroiding precision, a detector at a short
  sample-to-detector distance reads a structurally higher microstrain than the same precision
  would at a longer distance, simply because its usable rings sit at smaller pixel radii. A
  geometry visually confirmed against the rings (Step 3) with a modest overage on this default
  is a genuine, usable result with a refinement item noted, not a failure -- a *large* overage
  (tens of times the cap) together with a visibly drifting overlay is the real red flag.
- **`basin_check`** -- did the fit stay close to its own seed, or jump uncontrollably?
- **`cross_validation`** -- does a model fit on 90% of the rings predict the held-out 10%, or is
  it overfit to this one image?
- **`azimuth_coverage`** -- do the detected rings span enough of 360 degrees to actually
  constrain the tilt/distortion terms being fitted?

In [ ]:
print(f'Lsd:   {res.Lsd:.2f} um  ({res.Lsd/1000:.2f} mm)')
print(f'BC:    ({res.BC_y:.3f}, {res.BC_z:.3f}) px')
print(f'tilts: ty = {res.ty:+.4f} deg,  tz = {res.tz:+.4f} deg')
print(f'in-loop strain:      {res.in_loop_strain_uE:.2f} microstrain')
print(f'post-residual-map:   {res.post_residual_strain_uE:.2f} microstrain '
     f'(median {res.post_residual_strain_median_uE:.2f}, trimmed {res.post_residual_strain_trim_uE:.2f})')
print()
print('GATES (read every line -- a failed gate is not a footnote):')
for d in res.diagnostics:
    mark = {"ok": "PASS", "warn": "WARN", "fail": "FAIL"}.get(d.severity, d.severity.upper())
    print(f'  [{mark:5s}] {d.name}: {d.message}')


## Step 3 — Look at the rings: is the fit even looking at the right rings?

**A gate number by itself is not a check -- overlay the predicted rings on the image and look.**
Two calibrations can converge to the same strain and one can still be fitting the wrong rings
(a mis-indexed ring order, a ring the mask ate most of, a harmonic chasing detector artifacts
instead of the calibrant). This is the same "look at the picture before the summary" discipline
`05`'s Step 2 applies to claimed reflections, applied here to rings. Predicted ring radii come
from the material's own reflection conditions via `midas_hkls.centring_allowed` -- not a
hand-written extinction rule, which is exactly the kind of thing that goes silently wrong for
one material and not another (see that function's own docstring for a real incident).

In [ ]:
from midas_hkls import centring_allowed
import matplotlib.pyplot as plt

def predicted_ring_radii_px(a_ang, space_group, wavelength_A, lsd_um, px_um, max_radius_px, hmax=14):
    hkl = np.array([(h, k, l) for h in range(hmax+1) for k in range(hmax+1)
                    for l in range(hmax+1) if h+k+l > 0])
    hkl = hkl[centring_allowed(hkl, space_group)]
    d = a_ang / np.sqrt((hkl**2).sum(axis=1))          # cubic only -- matches this notebook's
    d = np.unique(np.round(d, 6))[::-1]                 # own (a,a,a,90,90,90) calibrant input
    radii = []
    for dd in d:
        s = wavelength_A / (2*dd)
        if s >= 1:
            continue
        tth = 2*np.arcsin(s)
        if tth >= np.pi/2:      # tan blows up / goes negative past 90 deg -- bogus radius
            continue
        r_px = (lsd_um*np.tan(tth)) / px_um
        if 0 < r_px <= max_radius_px:
            radii.append((np.degrees(tth), r_px))
    return radii

bcy, bcz = res.BC_y, res.BC_z
lsd_um = res.Lsd
max_r = min(bcy, img.shape[1]-bcy, bcz, img.shape[0]-bcz) * 1.3
radii = predicted_ring_radii_px(CALIBRANT_A, CALIBRANT_SPACE_GROUP, WAVELENGTH_A,
                                lsd_um, PIXEL_SIZE_UM, max_r)

disp = np.where(mask, np.nan, img)
vmax = np.nanpercentile(disp, 99.5)
fig, axes = plt.subplots(1, 2, figsize=(13, 6.2))
for a in axes:
    a.imshow(disp, cmap="viridis", vmin=0, vmax=vmax, origin="upper")
    for _tth, r in radii:
        a.add_patch(plt.Circle((bcy, bcz), r, fill=False, color="red", lw=0.7, alpha=0.85))
    a.plot([bcy], [bcz], "r+", ms=14)
    a.set_xlabel("column"); a.set_ylabel("row")
axes[0].set_title(f"predicted rings on the fitted geometry (Lsd={lsd_um/1000:.2f} mm)")
zoom = min(300, max_r*0.6)
axes[1].set_xlim(bcy-zoom, bcy+zoom); axes[1].set_ylim(bcz+zoom, bcz-zoom)
axes[1].set_title("zoom on the inner rings")
fig.suptitle(f"{len(radii)} predicted rings -- in-loop strain "
            f"{'PASSED' if res.in_loop_strain_uE <= 100 else 'FAILED'} "
            f"({res.in_loop_strain_uE:.1f} ue vs 100 ue cap)")
fig.tight_layout(); plt.show()

# the quantitative version of the same check: predicted vs. observed ring radius, per ring
yy, xx = np.mgrid[0:img.shape[0], 0:img.shape[1]]
rr = np.hypot(yy - bcz, xx - bcy).ravel()
vv = np.where(mask, 0, img).ravel()
nb = max(int(max_r), 1)
prof, edges = np.histogram(rr, bins=nb, range=(0, max_r), weights=vv)
cnt, _ = np.histogram(rr, bins=nb, range=(0, max_r))
prof = np.where(cnt > 0, prof/np.maximum(cnt, 1), 0)
ctr = 0.5*(edges[1:]+edges[:-1])
print(f"{'2theta (deg)':>13s} {'r_pred (px)':>12s} {'r_obs (px)':>11s} {'diff (px)':>10s} {'strain (ue)':>12s}")
for tth, r in radii[:12]:
    w = (ctr > r-8) & (ctr < r+8)
    if not w.any() or prof[w].max() <= 0:
        print(f"{tth:13.3f} {r:12.1f} {'--':>11s} {'--':>10s} {'--':>12s}")
        continue
    robs = float(ctr[w][np.argmax(prof[w])])
    strain_ue = abs(1 - robs/r) * 1e6
    print(f"{tth:13.3f} {r:12.1f} {robs:11.1f} {robs-r:+10.1f} {strain_ue:12.1f}")


## What this notebook does NOT do

- **It cannot see a detector roll** (rotation about the beam). A powder pattern from a single
  calibrant is invariant under that rotation by symmetry -- no amount of calibrant data fixes
  this. If a later residual map shows a clean radial pattern with a leftover tangential
  component, that is the signature to look for, and it needs single-crystal grain data to
  resolve (`midas-joint-ff-calibrate grain-tx`), not another powder exposure.
- **A gate failure here is not silently absorbed** -- but it is also not automatically a
  disaster. Look at the overlay before deciding: a modest `strain_cap` overage on a visually
  correct fit is a refinement item; a large one (tens of times the cap) with a visibly drifting
  overlay is a real wrong-basin failure, and that geometry should not be passed to `05`/`06`'s
  `Geometry` without resolving why.
- **It does not self-calibrate per raster position.** This is a single, one-time geometry for
  the whole raster; `05`/`06` take it as fixed. If you don't trust it for a specific position,
  `midas_defect.selfcal.selfcalibrate_from_crystals` is a separate, deliberate step.